In [14]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv('../data/dataset_clean_max.csv')
df.head()
df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v3.csv')

In [17]:
missing_percent = (df.isnull().sum() / len(df)) * 100
cols_to_drop = missing_percent[missing_percent > 85].index
df_train_clean = df.drop(columns=cols_to_drop)

# Garder que numériques + supprimer lignes NaN restantes
df_train_clean = df_train_clean.dropna()
df_train_clean = df_train_clean.select_dtypes(include=['int64', 'float64', 'bool'])
df_train_clean = df_train_clean.loc[:, df_train_clean.nunique() > 1]

X_train = df_train_clean.drop('target_is_fraud', axis=1)
y_train = df_train_clean['target_is_fraud']

model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Cross-validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')

# Entraînement
model.fit(X_train, y_train)

# Prédictions
y_train_pred = model.predict(X_train)

print("=== RÉSULTATS TRAIN ===")
print(f"CV Recall:  {cv_scores.mean():.4f}")
print(f"Accuracy:   {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Precision:  {precision_score(y_train, y_train_pred):.4f}")
print(f"Recall:     {recall_score(y_train, y_train_pred):.4f}")
print(f"F1-Score:   {f1_score(y_train, y_train_pred):.4f}")
print("\nMatrice de confusion:")
print(confusion_matrix(y_train, y_train_pred))

df_test_clean = df_test.drop(columns=cols_to_drop, errors='ignore')

customer_ids = df_test['customer_id'].copy()

df_test_clean = df_test_clean.fillna(df_test_clean.mean(numeric_only=True))

df_test_clean = df_test_clean.select_dtypes(include=['int64', 'float64', 'bool'])

for col in X_train.columns:
    if col not in df_test_clean.columns:
        df_test_clean[col] = 0

df_test_clean = df_test_clean[X_train.columns]

if 'target_is_fraud' in df_test_clean.columns:
    X_test = df_test_clean.drop('target_is_fraud', axis=1)
else:
    X_test = df_test_clean

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

results_df = pd.DataFrame({
    'customer_id': customer_ids,
    'target_is_fraud': y_test_pred,
    
})

results_df.to_csv('../data/DecisionTree_predictions.csv', index=False)
print(f"✓ {len(results_df)} prédictions sauvegardées")
print(results_df.head())

=== RÉSULTATS TRAIN ===
CV Recall:  0.9576
Accuracy:   0.9995
Precision:  0.9949
Recall:     0.9894
F1-Score:   0.9922

Matrice de confusion:
[[114142     18]
 [    38   3545]]
✓ 40000 prédictions sauvegardées
       customer_id  target_is_fraud
0  CUST_E5RX1BC9II                0
1  CUST_BHWIUKERGN                0
2  CUST_EXT9NA4CHU                0
3  CUST_9FSJE5R1NY                0
4  CUST_GDQXMODBED                0
